In [1]:
import cv2
import glob
import re
import easyocr
from ultralytics import YOLO

# =========================
# 1. YOLO MODEL
# =========================
model = YOLO(r"runs/detect/train-3/weights/best.pt")

# =========================
# 2. OCR MODEL (STABLE)
# =========================
reader = easyocr.Reader(['en'], gpu=True)

# =========================
# 3. DATA
# =========================
images = glob.glob(r"dataset/test/images/*.jpg")
print("IMAGES:", len(images))

# =========================
# 4. POST-PROCESSING
# =========================
def clean_plate(text):
    text = text.upper()
    text = re.sub(r'[^A-Z0-9]', '', text)

    replacements = {
        "O": "0",
        "I": "1",
        "Z": "2",
        "S": "5",
        "B": "8",
        "G": "6"
    }

    for k, v in replacements.items():
        text = text.replace(k, v)

    return text


def is_russian_plate(text):
    # базовая проверка РФ номера
    return 6 <= len(text) <= 9


# =========================
# 5. PIPELINE
# =========================
for img_path in images:

    img = cv2.imread(img_path)

    results = model.predict(img, conf=0.25, verbose=False)

    for r in results:
        for box in r.boxes:

            x1, y1, x2, y2 = map(int, box.xyxy[0])
            crop = img[y1:y2, x1:x2]

            # =========================
            # OCR (EasyOCR)
            # =========================
            ocr_result = reader.readtext(crop)

            text = ""

            if ocr_result:
                # берем самый уверенный результат
                text = max(ocr_result, key=lambda x: x[2])[1]

            # =========================
            # CLEAN
            # =========================
            text = clean_plate(text)

            if is_russian_plate(text):
                print(img_path, "→", text)

RuntimeError: module compiled against ABI version 0x1000009 but this version of numpy is 0x2000000

ImportError: numpy.core.multiarray failed to import